# BTL vs CDM: learning cycle detection, two cells at a time

**Status:** Session 1 only. These two cells construct exact probabilities; no sampled-data detection or model fitting has been done.

**Research goal:** Can we detect a strict pairwise preference cycle from noisy observations, distinguish it from sampling noise, and understand how graph structure affects estimation?

**Today's first question:** If A tends to beat B and B tends to beat C, must A tend to beat C?

## Cell 1 — derive the target before running code

Use $p_{ij}=P(i\mid\{i,j\})$, not the probability of a single joint ranking. The three probabilities below refer to **three different two-item menus**.

### BTL: one score for each item

$$p_{ij}=\frac{e^{\theta_i}}{e^{\theta_i}+e^{\theta_j}}
=\sigma(\theta_i-\theta_j),\qquad \sigma(t)=\frac{1}{1+e^{-t}}.$$

Take $(\theta_A,\theta_B,\theta_C)=(1,0,-1)$. These scores sum to zero; centering fixes the irrelevant common shift.

Before running Cell 2, calculate the three inputs to the sigmoid:

| Pair orientation | BTL logit: fill in by hand |
|---|---|
| A against B | $\theta_A-\theta_B=\;?$ |
| B against C | $\theta_B-\theta_C=\;?$ |
| C against A | $\theta_C-\theta_A=\;?$ |

The structural constraint is

$$(\theta_A-\theta_B)+(\theta_B-\theta_C)+(\theta_C-\theta_A)=0.$$

Thus the third logit is fixed once the first two are known. If the first two are positive, the third must be negative: a strict population majority three-cycle is impossible under BTL.

### CDM: an interaction for each ordered pair

Let $u_{xz}$ be the contribution of z's presence to x's utility. Then

$$v(x\mid S)=\sum_{z\in S\setminus\{x\}}u_{xz},\qquad
P(x\mid S)=\frac{e^{v(x\mid S)}}{\sum_{y\in S}e^{v(y\mid S)}}.$$

For $S=\{i,j\}$ this reduces to

$$p_{ij}=\frac{e^{u_{ij}}}{e^{u_{ij}}+e^{u_{ji}}}
=\sigma(u_{ij}-u_{ji}).$$

Choose rows as the **chosen/target item**, columns as the **other/context item**, in A, B, C order:

$$U=\begin{pmatrix}
0 & 1/2 & -1/2\\
-1/2 & 0 & 1/2\\
1/2 & -1/2 & 0
\end{pmatrix}.$$

Compute $u_{AB}-u_{BA}$, $u_{BC}-u_{CB}$ and $u_{CA}-u_{AC}$ yourself. Can all three be positive? Here the diagonal is unused, and the skew-symmetric matrix is a convenient construction, not a requirement on all CDMs. Pair-only data identify these differences, not every entry of U; centering alone does not remove that ambiguity.

## Our planned learning sequence

1. **Now:** exact BTL/CDM probabilities and the three-item example.
2. **Next:** majority graphs, graph direction, and a visible cycle witness.
3. Sample outcomes; build win counts, the undirected comparison graph, and the directed observed win graph.
4. Study connectivity, strong connectivity, finite BTL MLE, and separation with small counterexamples.
5. Build empirical and confidence-supported cycle detectors, including missing comparisons and ties.
6. Run repeated BTL controls and planted-CDM-cycle experiments with uncertainty estimates.
7. Compare fitted probabilities and held-out loss; study pairwise identifiability and larger-menu effects separately.

We will add only two cells per chat. The existing `01_Cycle_Detection_Design.ipynb` is a design reference, not completed simulation evidence.

**Distinctions to preserve:** an observed win-graph cycle is not proof of a population majority cycle; a population cycle rules out exact BTL representation but does not uniquely identify CDM. No detected cycle does not establish BTL. Ford's BTL criterion must not be applied unchanged to unrestricted CDM estimation.

**Sources:** supplied DA5453 Module 1 notes, Sections 2.3, 2.6 and 4.3–4.9; the supplied supervisor conversation; and [Seshadri, Peysakhovich & Ugander (2019), CDM definition](https://proceedings.mlr.press/v97/seshadri19a.html). The notes/transcript were supplied privately and are not copied into this repository.


In [1]:
# Cell 2 — check your hand calculations with exact model probabilities.
# No sampling, graph algorithms, or fitting yet.
from math import exp, isclose


def sigmoid(t):
    # Stable for both positive and negative t.
    return 1 / (1 + exp(-t)) if t >= 0 else exp(t) / (1 + exp(t))


items = ("A", "B", "C")
theta = (1.0, 0.0, -1.0)
U = ((0.0, 0.5, -0.5),
     (-0.5, 0.0, 0.5),
     (0.5, -0.5, 0.0))
pairs = ((0, 1), (1, 2), (2, 0))  # A over B, B over C, C over A
btl_logits = [theta[i] - theta[j] for i, j in pairs]
cdm_logits = [U[i][j] - U[j][i] for i, j in pairs]

print("Orientation     BTL logit   BTL probability   CDM logit   CDM probability")
for (i, j), b, c in zip(pairs, btl_logits, cdm_logits):
    print(f"{items[i]} over {items[j]}          {b:>5.1f}          {sigmoid(b):.4f}        {c:>5.1f}          {sigmoid(c):.4f}")

print("\nBTL triangle logit sum:", sum(btl_logits))
print("CDM triangle logit sum:", sum(cdm_logits))
print("All three oriented probabilities > 0.5?")
print("BTL:", all(sigmoid(t) > 0.5 for t in btl_logits))
print("CDM:", all(sigmoid(t) > 0.5 for t in cdm_logits))

assert isclose(sum(theta), 0.0)
assert isclose(sum(btl_logits), 0.0)
assert btl_logits == [1.0, 1.0, -2.0]
assert cdm_logits == [1.0, 1.0, 1.0]
for t in btl_logits + cdm_logits:
    assert isclose(sigmoid(t) + sigmoid(-t), 1.0)
print("\nChecks passed. Explain the last row before moving to graphs.")


Orientation     BTL logit   BTL probability   CDM logit   CDM probability
A over B            1.0          0.7311          1.0          0.7311
B over C            1.0          0.7311          1.0          0.7311
C over A           -2.0          0.1192          1.0          0.7311

BTL triangle logit sum: 0.0
CDM triangle logit sum: 3.0
All three oriented probabilities > 0.5?
BTL: False
CDM: True

Checks passed. Explain the last row before moving to graphs.
